# MLB Hitting vs. Pitching: Predicting Team Wins

**Research question:** How well can MLB team wins be predicted using hitting statistics compared with pitching/run-prevention statistics?

This is a regression project using MLB team-season data from 2000–2025. Seasons 2000–2022 are training data and 2023–2025 are held out for testing.


## Problem Definition

Target: team wins (`W`). Hitting predictors: runs per game, home runs per game, OBP, and SLG. Pitching predictors: runs allowed per game, ERA, and strikeouts per 9 innings. The project compares hitting-only, pitching-only, and combined models.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

url = 'https://raw.githubusercontent.com/corbtastik/lahman-baseball-db/master/Teams.csv'
df = pd.read_csv(url)
df = df[df['yearID'].between(2000, 2025)].copy()

df['runs_per_game'] = df['R'] / df['G']
df['runs_allowed_per_game'] = df['RA'] / df['G']
df['home_runs_per_game'] = df['HR'] / df['G']
df['obp'] = (df['H'] + df['BB'] + df['HBP']) / (df['AB'] + df['BB'] + df['HBP'] + df['SF'])
df['slg'] = (df['H'] + df['2B'] + 2*df['3B'] + 3*df['HR']) / df['AB']
df['so_per_9'] = df['SOA'] / (df['IPouts'] / 3) * 9

print('Observations:', len(df))
df.head()


## Data Description and Exploration

Each row is one MLB team-season. The dataset contains team wins, runs scored, batting totals, runs allowed, earned runs, strikeouts, and other team statistics.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(df['runs_per_game'], df['W'], alpha=.5)
axes[0].set(xlabel='Runs per Game', ylabel='Wins', title='Wins vs Runs/Game')
axes[1].scatter(df['runs_allowed_per_game'], df['W'], alpha=.5)
axes[1].set(xlabel='Runs Allowed/Game', ylabel='Wins', title='Wins vs Runs Allowed/Game')
axes[2].scatter(df['ERA'], df['W'], alpha=.5)
axes[2].set(xlabel='ERA', ylabel='Wins', title='Wins vs ERA')
plt.tight_layout()
plt.show()

cols = ['W','runs_per_game','home_runs_per_game','obp','slg','runs_allowed_per_game','ERA','so_per_9']
df[cols].corr()['W'].sort_values(ascending=False).round(3)


## Train/Test Split

A chronological split is used to avoid training on later seasons and then evaluating on earlier seasons. Training uses 2000–2022; testing uses 2023–2025.


In [ ]:
hitting = ['runs_per_game','home_runs_per_game','obp','slg']
pitching = ['runs_allowed_per_game','ERA','so_per_9']
combined = hitting + pitching
train = df[df['yearID'] <= 2022].copy()
test = df[df['yearID'] >= 2023].copy()
print(len(train), 'training observations')
print(len(test), 'testing observations')


## Baseline

The baseline predicts the average training-set number of wins for every test observation. The machine-learning models should improve on this benchmark.


In [ ]:
baseline = np.repeat(train['W'].mean(), len(test))
print('Baseline MAE:', round(mean_absolute_error(test['W'], baseline), 2))
print('Baseline RMSE:', round(np.sqrt(mean_squared_error(test['W'], baseline)), 2))
print('Baseline R2:', round(r2_score(test['W'], baseline), 3))


## Model Development and Evaluation

Two regression algorithms are compared: Linear Regression and K-Nearest Neighbors Regression. MAE and RMSE are errors in wins, so lower is better. R² is explanatory power, so higher is better.


In [ ]:
def score_model(model, features):
    model.fit(train[features], train['W'])
    pred = model.predict(test[features])
    return {'MAE': mean_absolute_error(test['W'], pred), 'RMSE': np.sqrt(mean_squared_error(test['W'], pred)), 'R2': r2_score(test['W'], pred)}, pred

rows = []
preds = {}
for name, features in [('Hitting', hitting), ('Pitching', pitching), ('Combined', combined)]:
    for model_name, model in [('Linear Regression', LinearRegression()), ('KNN Regression', KNeighborsRegressor(n_neighbors=7))]:
        metrics, pred = score_model(model, features)
        rows.append({'Features': name, 'Model': model_name, **metrics})
        preds[(name, model_name)] = pred

results = pd.DataFrame(rows).sort_values('RMSE')
results.round(3)


In [ ]:
best = preds[('Combined', 'KNN Regression')]
plt.figure(figsize=(6,5))
plt.scatter(test['W'], best, alpha=.65)
lo, hi = test['W'].min(), test['W'].max()
plt.plot([lo, hi], [lo, hi], linestyle='--')
plt.xlabel('Actual Wins')
plt.ylabel('Predicted Wins')
plt.title('Combined KNN: Actual vs Predicted Wins')
plt.show()


## Interpretation

The held-out results show that pitching/run prevention predicts wins somewhat better than the selected hitting variables when each group is modeled separately. The combined models perform substantially better than either group alone, showing that both offense and run prevention contain useful information about team wins.

This is a predictive result, not a causal claim. It does not prove that pitching alone causes more wins; model performance depends on the variables selected, the seasons used, and the modeling method.


## Limitations and Next Steps

The data are team-season aggregates and do not capture every factor affecting wins, including defense, baserunning, injuries, roster changes, strength of schedule, and bullpen usage. The test period is also only three seasons. Future work could add defensive/baserunning variables, use cross-validation, and compare additional algorithms such as Random Forest or Gradient Boosting.


## Conclusion

For these 2000–2025 MLB team-season data, pitching/run prevention produced better held-out win predictions than the selected hitting variables when modeled separately. However, combining hitting and pitching produced the strongest predictions, indicating that both sides of the game are important for explaining and predicting team wins.


## AI / Code Transparency

Generative AI was used to assist with project planning, research-question refinement, Python code structure, model setup, interpretation, and written explanations. The student should review and understand all code and results before submission and follow the course requirements for AI disclosure.
